# 03 — Model Training & Evaluation

**Objective:** Train both the Bi-LSTM Autoencoder (unsupervised) and XGBoost (supervised), evaluate each model, build the hybrid ensemble, and achieve target metrics.

**Target Metrics:**
| Model | ROC-AUC | PR-AUC | F1 |
|---|---|---|---|
| Bi-LSTM Autoencoder | ≥ 0.90 | ≥ 0.80 | ≥ 0.85 |
| XGBoost baseline | ≥ 0.95 | ≥ 0.80 | ≥ 0.85 |

**Sections:**
1. Load processed data
2. Train Bi-LSTM Autoencoder
3. Evaluate autoencoder
4. Train XGBoost
5. Evaluate XGBoost
6. Hybrid Ensemble evaluation
7. Summary

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.dataset import load_splits
from src.models.autoencoder import build_bilstm_autoencoder, summarize
from src.models.trainer import train_autoencoder, compute_threshold, get_reconstruction_errors
from src.models.evaluator import full_evaluation, evaluate_ensemble
from src.models.xgboost_model import train_xgboost, plot_feature_importance, hybrid_predict

sns.set_style("darkgrid")
os.makedirs("../logs", exist_ok=True)
os.makedirs("../models", exist_ok=True)
print("Imports OK")

Imports OK


## 1. Load Processed Data

In [2]:
splits = load_splits("../data/processed")

X_train = splits["X_train"]
y_train = splits["y_train"]
X_val   = splits["X_val"]
y_val   = splits["y_val"]
X_test  = splits["X_test"]
y_test  = splits["y_test"]

# Val subset — only normals for autoencoder threshold
X_val_normal = X_val[y_val == 0]

# Advanced features for XGBoost
X_train_adv = np.load("../data/features/X_train_advanced.npy")
X_val_adv   = np.load("../data/features/X_val_advanced.npy")
X_test_adv  = np.load("../data/features/X_test_advanced.npy")

# XGBoost needs the full val labels (including anomalies)
print(f"X_train      : {X_train.shape}   y_train anomalies: {y_train.sum()} (must be 0)")
print(f"X_val        : {X_val.shape}     y_val   anomalies: {y_val.sum()}")
print(f"X_test       : {X_test.shape}    y_test  anomalies: {y_test.sum()}")
print(f"X_val_normal : {X_val_normal.shape}")
print(f"X_train_adv  : {X_train_adv.shape}")

  Loaded arrays: ['X_test', 'X_train', 'X_val', 'y_test', 'y_train', 'y_val']
X_train      : (26565, 60, 50)   y_train anomalies: 0 (must be 0)
X_val        : (6000, 60, 50)     y_val   anomalies: 294
X_test       : (6000, 60, 50)    y_test  anomalies: 271
X_val_normal : (5706, 60, 50)
X_train_adv  : (26565, 1050)


## 2. Train Bi-LSTM Autoencoder

**Training protocol:**
- Input == Target (reconstruction objective)
- Training only on normal sequences
- EarlyStopping (patience=10), ReduceLROnPlateau, ModelCheckpoint

In [3]:
model = build_bilstm_autoencoder(seq_length=60, n_features=50)
summarize(model)

Model: "BiLSTM_Autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sensor_input (InputLayer)       │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_bilstm_1 (Bidirectional)    │ (None, 60, 256)        │       183,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_drop_1 (Dropout)            │ (None, 60, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_bilstm_2 (Bidirectional)    │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_drop_2 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat (RepeatVector)           │ (None, 60, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_lstm_1 (LSTM)               │ (None, 60, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_drop_1 (Dropout)            │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_lstm_2 (LSTM)               │ (None, 60, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction                  │ (None, 60, 50)         │         6,450 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 481,874 (1.84 MB)

 Trainable params: 481,874 (1.84 MB)

 Non-trainable params: 0 (0.00 B)


  Total trainable parameters: 481,874


In [4]:
history = train_autoencoder(
    model,
    X_train_clean=X_train,          # normal-only (filtered by preprocessor)
    X_val_normal=X_val_normal,       # normal-only val subset
    epochs=100,
    batch_size=32,
    model_path="../models/bilstm_autoencoder.h5",
)


  Training on 26,565 normal sequences…
  Validation on 5,706 normal sequences…
Epoch 1/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - loss: 0.7976
Epoch 1: val_loss improved from None to 0.52617, saving model to ../models/bilstm_autoencoder.h5


831/831 ━━━━━━━━━━━━━━━━━━━━ 97s 112ms/step - loss: 0.6822 - val_loss: 0.5262 - learning_rate: 0.0010
Epoch 2/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 0.5323
Epoch 2: val_loss improved from 0.52617 to 0.52132, saving model to ../models/bilstm_autoencoder.h5


831/831 ━━━━━━━━━━━━━━━━━━━━ 94s 113ms/step - loss: 0.5304 - val_loss: 0.5213 - learning_rate: 0.0010
Epoch 3/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 0.5271
Epoch 3: val_loss improved from 0.52132 to 0.51950, saving model to ../models/bilstm_autoencoder.h5


831/831 ━━━━━━━━━━━━━━━━━━━━ 95s 114ms/step - loss: 0.5264 - val_loss: 0.5195 - learning_rate: 0.0010
Epoch 4/100
830/831 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - loss: 0.5252
Epoch 4: val_loss improved from 0.51950 to 0.51948, saving model to ../models/bilstm_autoencoder.h5


831/831 ━━━━━━━━━━━━━━━━━━━━ 83s 99ms/step - loss: 0.5248 - val_loss: 0.5195 - learning_rate: 0.0010
Epoch 5/100
 76/831 ━━━━━━━━━━━━━━━━━━━━ 38s 51ms/step - loss: 0.5241

KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history["loss"],     label="Train Loss", color="#2196F3")
ax.plot(history.history["val_loss"], label="Val Loss",   color="#F44336")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Autoencoder Training Curve", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("../logs/training/training_curve.png", dpi=120, bbox_inches="tight")
plt.show()
os.makedirs("../logs/training", exist_ok=True)

## 3. Compute Threshold & Evaluate Autoencoder

**Threshold:** 95th percentile of reconstruction error on normal val sequences.  
Sequences with MSE > threshold are classified as anomalies.

In [ ]:
threshold = compute_threshold(
    model,
    X_val_normal,
    percentile=95,
    save_path="../models/threshold.pkl",
)

In [ ]:
ae_results = full_evaluation(
    model, X_test, y_test, threshold,
    save_dir="../logs",
    fig_name="autoencoder_eval.png",
)

In [ ]:
errors = get_reconstruction_errors(model, X_test)
normal_errors  = errors[y_test == 0]
anomaly_errors = errors[y_test == 1]

# CRITICAL ASSERTION: Anomaly error must exceed normal error
assert np.mean(anomaly_errors) > np.mean(normal_errors), \
    "🚨 INVERTED MODEL: normal_error > anomaly_error. Check training data."
print(f"✅ Error direction correct:")
print(f"   Normal  error mean: {np.mean(normal_errors):.6f}")
print(f"   Anomaly error mean: {np.mean(anomaly_errors):.6f}")
print(f"   Separation:         {np.mean(anomaly_errors) - np.mean(normal_errors):.6f}")

## 4. Train XGBoost Supervised Baseline

**Note:** XGBoost uses ALL training data (normal + anomaly) with supervised labels, unlike the autoencoder.

In [ ]:
# For XGBoost, we need the full training set including anomalies
# Load original (pre-filtering) train via the raw splits  
X_raw = np.load("../data/raw/X.npy")
y_raw = np.load("../data/raw/y.npy")

# Re-apply the same temporal split to get the full (unfiltered) train with labels
n = len(X_raw)  
i_train = int(n * 0.70)
i_val   = int(n * 0.85)

import joblib
scaler = joblib.load("../models/scaler.pkl")

def scale_3d(X_3d, scaler):
    n, t, f = X_3d.shape
    return scaler.transform(X_3d.reshape(-1, f)).reshape(n, t, f).astype(np.float32)

X_train_full = scale_3d(X_raw[:i_train], scaler)
y_train_full = y_raw[:i_train]

print(f"XGBoost training set: {X_train_full.shape}  anomalies={y_train_full.sum()}")

# Build features for full train
from src.features.feature_pipeline import build_feature_matrix
print("Building feature matrix for full train (with anomalies)…")
X_train_full_adv = build_feature_matrix(X_train_full, verbose=True)

In [ ]:
xgb_model = train_xgboost(
    X_train_full_adv, y_train_full,
    X_val_adv, y_val,
    save_path="../models/xgboost_baseline.json",
)

## 5. Evaluate XGBoost

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

xgb_proba = xgb_model.predict_proba(X_test_adv)[:, 1]
xgb_pred  = xgb_model.predict(X_test_adv)

roc = roc_auc_score(y_test, xgb_proba)
pr  = average_precision_score(y_test, xgb_proba)

print("\n" + "═"*55)
print("  XGBOOST EVALUATION — TEST SET")
print("═"*55)
print(f"  ROC-AUC : {roc:.4f}  {'✅' if roc >= 0.95 else '⚠️'}  (target ≥ 0.95)")
print(f"  PR-AUC  : {pr:.4f}   {'✅' if pr >= 0.80 else '⚠️'}  (target ≥ 0.80)")
print(f"\n{classification_report(y_test, xgb_pred, target_names=['Normal','Anomaly'])}")

In [ ]:
plot_feature_importance(
    xgb_model, top_n=20,
    save_path="../logs/xgb_feature_importance.png",
)

## 6. Hybrid Ensemble (60% XGBoost + 40% Autoencoder)

In [ ]:
from src.models.evaluator import evaluate_ensemble

ensemble_scores = hybrid_predict(
    autoencoder=model,
    xgb_model=xgb_model,
    X_sequences=X_test,
    X_features=X_test_adv,
    alpha=0.6,
)

ens_results = evaluate_ensemble(errors, xgb_proba, y_test, alpha=0.6)

## 7. Summary

In [ ]:
print("\n" + "═"*60)
print("  FINAL MODEL COMPARISON")
print("═"*60)
print(f"  {'Model':<25} {'ROC-AUC':>10} {'PR-AUC':>10}  {'Status':>12}")
print("  " + "-"*57)
print(f"  {'Bi-LSTM Autoencoder':<25} {ae_results['roc_auc']:>10.4f} {ae_results['pr_auc']:>10.4f}  "
      f"{'✅' if ae_results['roc_auc']>=0.90 else '⚠️'} AE")
print(f"  {'XGBoost Baseline':<25} {roc:>10.4f} {pr:>10.4f}  "
      f"{'✅' if roc>=0.95 else '⚠️'} XGB")
print(f"  {'Hybrid Ensemble':<25} {ens_results['roc_auc']:>10.4f} {ens_results['pr_auc']:>10.4f}  "
      f"{'✅' if ens_results['roc_auc']>=0.90 else '⚠️'} ENS")
print("═"*60)
print("\n✅ Artefacts saved:")
print("   models/bilstm_autoencoder.h5")
print("   models/xgboost_baseline.json")
print("   models/threshold.pkl")
print("   models/scaler.pkl")
print("\n→ NEXT STEP: Run notebook 04_kafka_test.ipynb")